In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
df = pd.read_csv("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv")

pd.set_option('display.max_columns',None)
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Data Understanding

In [ ]:
df.info()
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'],errors='coerce')
df['TotalCharges'] = df['TotalCharges'].astype(float)

In [ ]:
df.hist()

In [ ]:
df.head()

# Feature Engineering

In [ ]:
df.dropna(inplace=True)

In [ ]:
# Checking Unique features in each columns
def check_unique():
    col_unique = [col for col in df.columns if col != 'customerID']
    for col in col_unique:
        uni = df[col].unique()
        print(f"{col} : {uni} ")
    # print('\n')
check_unique()

In [ ]:
# Encoding
# Dense = storing every empty seat in a stadium.
# Sparse = storing only seat numbers that are occupied.

# If stadium is huge → sparse wins.
# If small room → doesn’t matter.from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OneHotEncoder

nominal_cols = [
    'gender',
    'Partner',
    'Dependents',
    'PhoneService',
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'PaperlessBilling',
    'PaymentMethod'
]

# ordinal encoding and target var
df['Churn']=df['Churn'].map({'Yes': 1, 'No': 0})
df['Contract']=df['Contract'].map({'One year': 1, 'Month-to-month': 0, 'Two year': 2})

# nominal encoding
ohe = OneHotEncoder(drop='first',sparse_output=False)
encoded = ohe.fit_transform(df[nominal_cols])
encorded_df = pd.DataFrame(encoded,columns=ohe.get_feature_names_out(nominal_cols),index=df.index)
print(df.shape)
df.drop(columns=nominal_cols,inplace=True)
print(df.shape)
main_df = pd.concat([df,encorded_df],axis=1)

In [ ]:
main_df.head()

In [ ]:
main_df.isnull().sum()

In [ ]:
# Handling imbalance Dataset
main_df['Churn'].value_counts()
from sklearn.model_selection import train_test_split

X = main_df.drop(columns=['customerID','Churn'])
y = df['Churn']

X_train,X_test,Y_train,Y_test=train_test_split(X,y,stratify=y,random_state=42)

# Baseline Models

## Logistic Regrestion

In [ ]:
# Train
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

lr = LogisticRegression(class_weight="balanced")
lr.fit(X_train,Y_train)
lr_p = lr.predict(X_test)
cm = confusion_matrix(Y_test, lr_p)
print("Confusion Matrix:")
print(cm)

#Evaluation
from sklearn.metrics import classification_report, confusion_matrix
cls_rpt_lr = classification_report(Y_test,lr_p)
print('--------------Logistic Regrestion classification_report---------------')
print(cls_rpt_lr)

## Random Forest

In [ ]:
# Train
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train,Y_train)
rf_p = rf.predict(X_test)
cm = confusion_matrix(Y_test, rf_p)
print("Confusion Matrix:")
print(cm)

#Evaluation
cls_rpt_rf = classification_report(Y_test,rf_p)
print('--------------RandomForestClassifier classification_report---------------')
print(cls_rpt_rf)

## XG boost

In [ ]:
from xgboost import XGBClassifier

# scale_pos_weight = negative_class / positive_class

# Train
xg = XGBClassifier(
    scale_pos_weight=5174/1869,
    n_estimators=300,
    max_depth=5,
    learning_rate=0.01,
    eval_metric="logloss",
    random_state=42,
    colsample_bytree=1.0,
    subsample = 0.6
)
xg.fit(X_train,Y_train)
xg_p = xg.predict(X_test)
cm = confusion_matrix(Y_test, xg_p)
print("Confusion Matrix:")
print(cm)

#Evaluation
cls_rpt_xg = classification_report(Y_test,xg_p)
print('--------------XGBClassifier classification_report---------------')
print(cls_rpt_xg)

# Hyperparameter Tuning

## Grid Search CV

In [ ]:
# GridSearchCV
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=xg,
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, Y_train)

print("Best Params:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)
#i will use best param in my orignal model

## Random Search CV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [200, 300, 500, 700],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3, 0.5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}


random_search = RandomizedSearchCV(
    estimator=xg,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, Y_train)

print("Best Params:", random_search.best_params_)
print("Best CV Score:", random_search.best_score_)
#i will use best param in my orignal model

In [ ]:
import joblib as jb

jb.dump(xg,'HyperParamXGBoost.pkl')

# MLOPS Building in next part